# Azure AI Foundry Agents with Foundry IQ

This notebook demonstrates the **Microsoft Agent Framework** with **Foundry IQ** for intelligent retrieval:

## Retrieval Modes
- **Semantic Mode**: Hybrid search with semantic ranking - fast, deterministic
- **Agentic Mode**: AI-powered retrieval with query planning and multi-step search

## Components
- `AzureAIAgentClient` - Creates agents in Azure AI Foundry
- `AzureAISearchContextProvider` - Provides grounding via Azure AI Search
- `ChatAgent` - Orchestrates conversations with context providers

## References
- [Foundry IQ Agent Framework Integration](https://devblogs.microsoft.com/foundry/foundry-iq-agent-framework-integration/)
- [Azure AI Foundry Agents](https://learn.microsoft.com/en-us/agent-framework/user-guide/agents/agent-types/azure-ai-foundry-agent)

In [ ]:
# Install the agent framework packages if needed
# !pip install agent-framework-azure-ai --pre
# !pip install azure-identity

# =============================================================================
# IMPORTANT: Authentication for Agent Framework vs Agent Identities
# =============================================================================
# 
# This notebook uses the Microsoft Agent Framework to create chat agents.
# The framework requires authentication to Azure AI Foundry services.
#
# ⚠️  COMMON ERROR: AADSTS82001
# If you see: "Agentic application '...' is not permitted to request app-only 
# tokens for resource '...'"
#
# This happens when DefaultAzureCredential picks up an Agent Identity context
# (from environment variables or managed identity). Agent Identities use a 
# DIFFERENT token flow and cannot request app-only tokens directly.
#
# SOLUTIONS:
# 1. Use InteractiveBrowserCredential (this notebook's approach)
# 2. Use AzureCliCredential if you're logged in via `az login`
# 3. Clear any AZURE_CLIENT_ID env vars that point to agent identities
# 4. For production: Use Microsoft Entra SDK for Agent ID container
#
# See: https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/microsoft-entra-sdk-for-agent-identities
# =============================================================================

In [ ]:
import os
import sys
import yaml
import warnings
import logging
from pathlib import Path
from dotenv import load_dotenv

# =============================================================================
# Load .env FIRST with override=True to ensure we use .env values
# =============================================================================
# override=True ensures .env values take precedence over any shell/system env vars
load_dotenv(override=True)

print("✅ Environment loaded from .env (override=True)")
print(f"   AZURE_TENANT_ID: {os.getenv('AZURE_TENANT_ID')[:20]}..." if os.getenv('AZURE_TENANT_ID') else "   ❌ AZURE_TENANT_ID: NOT SET")
print(f"   AZURE_CLIENT_ID: {os.getenv('AZURE_CLIENT_ID')[:20]}..." if os.getenv('AZURE_CLIENT_ID') else "   ℹ️  AZURE_CLIENT_ID: Not set (OK for Path B)")

# =============================================================================
# Suppress noisy warnings BEFORE importing Azure/aiohttp libraries
# =============================================================================
# These warnings are cosmetic and don't affect functionality
warnings.filterwarnings("ignore", message=".*Unclosed.*")
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", message=".*is not a known attribute.*")

# Suppress Azure SDK logging noise
logging.getLogger("azure").setLevel(logging.ERROR)
logging.getLogger("aiohttp").setLevel(logging.ERROR)

# Redirect stderr for aiohttp cleanup messages (they're printed directly, not through warnings)
class StderrFilter:
    """Filter stderr to suppress aiohttp/asyncio noise."""
    def __init__(self, stream):
        self.stream = stream
        self.suppress_patterns = [
            'Unclosed client session',
            'Unclosed connector',
            'k_nearest_neighbors is not a known attribute',
        ]
    
    def write(self, msg):
        if not any(pattern in msg for pattern in self.suppress_patterns):
            self.stream.write(msg)
    
    def flush(self):
        self.stream.flush()

sys.stderr = StderrFilter(sys.__stderr__)

# Microsoft Agent Framework
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient, AzureAISearchContextProvider

# Azure Identity - Use synchronous InteractiveBrowserCredential
# NOTE: DefaultAzureCredential can pick up agent identity context which causes 
# AADSTS82001 errors because agent identities require special token flows.
# InteractiveBrowserCredential authenticates you as a human user via browser device code flow.
from azure.identity import InteractiveBrowserCredential

print("")
print("✅ Imports loaded successfully")
print("🔇 Noisy warnings suppressed (aiohttp, k_nearest_neighbors)")

In [ ]:
# =============================================================================
# Load Agent Configurations from YAML
# =============================================================================

def load_agent_configs(agents_dir: Path) -> dict:
    """Load all agent configurations from YAML files."""
    configs = {}
    
    if not agents_dir.exists():
        print(f"⚠️ Agents directory not found: {agents_dir}")
        return configs
    
    for yaml_file in agents_dir.glob('*.yaml'):
        try:
            with open(yaml_file, 'r') as f:
                config = yaml.safe_load(f)
                agent_name = config.get('name')
                if agent_name:
                    configs[agent_name] = config
                    # Get first index name for display
                    indexes = config.get('tools', {}).get('azure_ai_search', {}).get('indexes', [])
                    index_name = indexes[0].get('name') if indexes else 'none'
                    print(f"  ✅ {agent_name} → {index_name}")
        except Exception as e:
            print(f"  ❌ Error loading {yaml_file.name}: {e}")
    
    return configs

print("Loading agent configurations...")
print("-" * 50)
agent_configs = load_agent_configs(AGENTS_DIR)
print(f"\n📦 Loaded {len(agent_configs)} agent configurations")

## Semantic Mode Retrieval

**Semantic mode** performs hybrid search with semantic ranking:
- Combines keyword + vector search
- Returns top-k documents ranked by relevance
- Fast and deterministic
- Best for: Simple queries with direct answers in documents

In [ ]:
# =============================================================================
# Credential Helper - Import from shared utils.py
# =============================================================================
# Uses InteractiveBrowserCredential with fallback to DeviceCodeCredential.
# Credential is cached to avoid repeated auth prompts.

from utils import get_user_credential, get_graph_token, get_agent_credential

# Get tenant ID from config or environment
TENANT_ID = os.getenv('AZURE_TENANT_ID') or "9249ded8-dff5-4e90-9d80-3ae45c13ec3f"

print("✅ Credential helpers loaded from utils.py")
print(f"   TENANT_ID: {TENANT_ID}")
print("\n   Available functions:")
print("   • get_user_credential() - Interactive browser auth (cached)")
print("   • get_graph_token(config) - Graph API token via MSAL")
print("   • get_agent_credential(tenant, blueprint_id, secret) - Agent identity")


In [ ]:
# =============================================================================
# Identity Diagnostics - Concise Summary
# =============================================================================
import json

def show_identity_info():
    """Display identity configuration summary."""
    
    print("🔐 Identity Configuration")
    print("=" * 50)
    
    # Environment variables
    env_vars = {
        'AZURE_CLIENT_ID': os.getenv('AZURE_CLIENT_ID'),
        'AZURE_TENANT_ID': os.getenv('AZURE_TENANT_ID'),
        'AZURE_CLIENT_SECRET': '***' if os.getenv('AZURE_CLIENT_SECRET') else None,
    }
    
    print("\n📋 Environment Variables:")
    for var, val in env_vars.items():
        status = "✅" if val else "❌"
        display = val[:20] + "..." if val and len(val) > 20 and '***' not in val else val
        print(f"   {status} {var}: {display or 'Not set'}")
    
    # a365 config
    print("\n📋 Agent Blueprint Config:")
    if Path('../a365.generated.config.json').exists():
        with open('../a365.generated.config.json') as f:
            gen = json.load(f)
        print(f"   Blueprint ID: {gen.get('agentBlueprintId', 'N/A')}")
        print(f"   Service Principal: {gen.get('agentBlueprintServicePrincipalObjectId', 'N/A')}")
        print(f"   Secret: {'✅ Set' if gen.get('agentBlueprintClientSecret') else '❌ Not set'}")
    else:
        print("   ⚠️  a365.generated.config.json not found")
    
    # Foundry config
    print("\n📋 Foundry Config:")
    if Path('foundry-config.json').exists():
        with open('foundry-config.json') as f:
            fc = json.load(f)
        foundry = fc.get('foundry', {})
        print(f"   Project: {foundry.get('project_name', 'N/A')}")
        print(f"   Endpoint: {foundry.get('endpoint', 'N/A')[:50]}...")
    else:
        print("   ⚠️  foundry-config.json not found")

show_identity_info()


In [ ]:
# =============================================================================
# Verify Current User Identity
# =============================================================================
import jwt  # PyJWT library

async def verify_current_identity():
    """
    Get a token and decode it to see who we're authenticated as.
    This helps verify we're using the right identity for queries.
    """
    print("🔍 Verifying Current Identity...")
    print("-" * 50)
    
    credential = get_user_credential()
    
    try:
        # Get a token for Azure management (common scope)
        token = credential.get_token("https://management.azure.com/.default")
        
        # Decode the token (without verification) to see claims
        # Note: In production, always verify tokens properly
        decoded = jwt.decode(token.token, options={"verify_signature": False})
        
        print(f"✅ Successfully authenticated!")
        print(f"")
        print(f"   👤 User/App Info:")
        print(f"      • Name: {decoded.get('name', 'N/A')}")
        print(f"      • UPN/Email: {decoded.get('upn') or decoded.get('unique_name') or decoded.get('preferred_username', 'N/A')}")
        print(f"      • Object ID (oid): {decoded.get('oid', 'N/A')}")
        print(f"      • App ID (appid/azp): {decoded.get('appid') or decoded.get('azp', 'N/A')}")
        print(f"")
        print(f"   🏢 Tenant Info:")
        print(f"      • Tenant ID: {decoded.get('tid', 'N/A')}")
        print(f"      • Issuer: {decoded.get('iss', 'N/A')}")
        print(f"")
        print(f"   🔐 Token Type:")
        print(f"      • Identity Type (idtyp): {decoded.get('idtyp', 'user (default)')}")
        print(f"      • Subject (sub): {decoded.get('sub', 'N/A')[:20]}...")
        
        # Check for agent-specific claims
        if decoded.get('xms_act_fct') or decoded.get('xms_sub_fct'):
            print(f"")
            print(f"   ⚠️  Agent Identity Claims Detected:")
            print(f"      • Actor Facet: {decoded.get('xms_act_fct', 'N/A')}")
            print(f"      • Subject Facet: {decoded.get('xms_sub_fct', 'N/A')}")
        else:
            print(f"")
            print(f"   ✅ No agent identity claims - this is a regular user token")
            
        return decoded
        
    except Exception as e:
        print(f"❌ Failed to get/decode token: {e}")
        return None

# Run verification
await verify_current_identity()

In [ ]:
# =============================================================================
# Validate Entra Agent Identity Association
# =============================================================================
# Verifies agent identity configuration and checks Foundry agent status.

import json
import requests
from pathlib import Path
from azure.ai.projects import AIProjectClient

async def validate_agent_identity_setup():
    """Validate Entra Agent Identity configuration."""
    
    print("🔐 Agent Identity Validation")
    print("=" * 60)
    
    # Load configurations
    a365_config = {}
    gen_config = {}
    
    if Path('../a365.config.json').exists():
        with open('../a365.config.json') as f:
            a365_config = json.load(f)
    
    if Path('../a365.generated.config.json').exists():
        with open('../a365.generated.config.json') as f:
            gen_config = json.load(f)
    
    blueprint_id = gen_config.get('agentBlueprintId')
    sp_id = gen_config.get('agentBlueprintServicePrincipalObjectId')
    has_secret = bool(gen_config.get('agentBlueprintClientSecret'))
    
    # Summary table
    print(f"""
Configuration Status:
  Blueprint ID:      {blueprint_id or '❌ Not configured'}
  Service Principal: {sp_id or '❌ Not configured'}
  Client Secret:     {'✅ Configured' if has_secret else '❌ Not set'}
  Identity Name:     {a365_config.get('agentIdentityDisplayName', 'N/A')}
""")
    
    # Verify service principal in Entra ID
    if sp_id:
        try:
            credential = get_user_credential()
            token = credential.get_token("https://graph.microsoft.com/.default")
            headers = {'Authorization': f'Bearer {token.token}'}
            
            response = requests.get(
                f"https://graph.microsoft.com/v1.0/servicePrincipals/{sp_id}",
                headers=headers
            )
            
            if response.status_code == 200:
                sp = response.json()
                print(f"✅ Service Principal verified: {sp.get('displayName')}")
            else:
                print(f"⚠️  Could not verify SP: {response.status_code}")
        except Exception as e:
            print(f"⚠️  Graph API error: {e}")
    
    # List Foundry agents
    try:
        credential = get_user_credential()
        project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
        agents = list(project_client.agents.list())
        
        if agents:
            print(f"\n📋 Foundry Agents ({len(agents)}):")
            for agent in agents:
                print(f"   • {agent.name} (id: {agent.id})")
        else:
            print("\n📋 No Foundry agents found")
    except Exception as e:
        print(f"⚠️  Could not list agents: {e}")
    
    # Usage guidance
    print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ℹ️  Current Status: Agents use CALLER identity (your token)

To use agent blueprint identity instead:

  from utils import get_agent_credential
  
  agent_cred = get_agent_credential(
      tenant_id="{a365_config.get('tenantId', 'TENANT_ID')}",
      blueprint_id="{blueprint_id or 'BLUEPRINT_ID'}",
      client_secret="<from a365.generated.config.json>"
  )
  project_client = AIProjectClient(endpoint=..., credential=agent_cred)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

await validate_agent_identity_setup()


In [ ]:
# =============================================================================
# Semantic Mode: Query with Hybrid Search + Semantic Ranking
# =============================================================================
import textwrap
import re

def extract_source_references(answer_text: str) -> list[str]:
    """
    Extract source references mentioned in the answer text.
    The model typically references sources as [Source: name] or similar patterns.
    
    Returns a list of unique source identifiers found in the text.
    """
    sources = []
    seen = set()
    
    # Pattern 1: [Source: xxx] or (Source: xxx)
    pattern1 = r'[\[\(]Source:\s*([^\]\)]+)[\]\)]'
    for match in re.finditer(pattern1, answer_text, re.IGNORECASE):
        src = match.group(1).strip()
        if src not in seen:
            seen.add(src)
            sources.append(src)
    
    # Pattern 2: "Source: xxx" in quotes
    pattern2 = r'"Source:\s*([^"]+)"'
    for match in re.finditer(pattern2, answer_text, re.IGNORECASE):
        src = match.group(1).strip()
        if src not in seen:
            seen.add(src)
            sources.append(src)
    
    # Pattern 3: Document references like [doc1], [source1†filename.pdf]
    pattern3 = r'\[([a-zA-Z0-9_-]+(?:†[^\]]+)?)\]'
    for match in re.finditer(pattern3, answer_text):
        src = match.group(1).strip()
        # Filter out common non-source brackets
        if src.lower() not in ['source', 'note', 'citation', 'ref'] and src not in seen:
            # Check if it looks like a source reference (has numbers, †, or looks like a filename)
            if re.search(r'\d|†|\.pdf|\.md', src, re.IGNORECASE):
                seen.add(src)
                sources.append(src)
    
    # Pattern 4: Footnote style [^1^], [^2^], etc.
    pattern4 = r'\[\^(\d+)\^\]'
    for match in re.finditer(pattern4, answer_text):
        src = f"Document {match.group(1)}"
        if src not in seen:
            seen.add(src)
            sources.append(src)
    
    return sources

def extract_citations(result):
    """
    Extract citation annotations from agent response if available.
    Returns a list of citation info dicts.
    """
    citations = []
    seen_titles = set()
    
    if hasattr(result, 'messages'):
        for message in result.messages:
            if hasattr(message, 'contents'):
                for content in message.contents:
                    if hasattr(content, 'annotations') and content.annotations:
                        for annotation in content.annotations:
                            # Check if it's a citation annotation
                            if hasattr(annotation, 'type') and annotation.type == 'citation':
                                title = getattr(annotation, 'title', None)
                                url = getattr(annotation, 'url', None)
                                filepath = getattr(annotation, 'filepath', None)
                                
                                # Deduplicate by title
                                if title and title not in seen_titles:
                                    seen_titles.add(title)
                                    citations.append({
                                        'title': title,
                                        'url': url,
                                        'filepath': filepath,
                                    })
    return citations

def format_answer_with_sources(answer: str, citations: list = None, width: int = 56):
    """Format the answer with wrapped text and source citations."""
    print("\n" + "=" * 60)
    print("✅ ANSWER")
    print("=" * 60)
    
    # Wrap text for better readability
    wrapped = textwrap.fill(answer, width=width)
    for line in wrapped.split('\n'):
        print(f"   {line}")
    
    # Show structured citations if available
    if citations:
        print("\n" + "-" * 60)
        print("📚 SOURCES (Structured Citations)")
        print("-" * 60)
        for i, citation in enumerate(citations, 1):
            title = citation.get('title', 'Unknown')
            url = citation.get('url')
            filepath = citation.get('filepath')
            
            print(f"   [{i}] {title}")
            if url:
                print(f"       🔗 {url}")
            if filepath:
                print(f"       📁 {filepath}")
    
    # Also extract inline source references from the answer
    inline_sources = extract_source_references(answer)
    if inline_sources:
        print("\n" + "-" * 60)
        print("📑 REFERENCED SOURCES (from answer text)")
        print("-" * 60)
        for i, src in enumerate(inline_sources, 1):
            print(f"   [{i}] {src}")
    
    print("=" * 60 + "\n")

async def query_semantic_mode(
    query: str,
    index_name: str,
    instructions: str = "You are a helpful assistant. Answer based on the provided context. Always mention which source document you got the information from.",
    show_sources: bool = True
):
    """
    Query using semantic mode - hybrid search with semantic ranking.
    
    This mode:
    - Performs keyword + vector hybrid search
    - Uses semantic ranking to reorder results
    - Returns top_k most relevant documents
    - Fast and deterministic
    
    Args:
        show_sources: If True, display source documents used for the answer
    """
    # Header
    print("\n" + "=" * 60)
    print("🔍 SEMANTIC MODE QUERY")
    print("=" * 60)
    print(f"   📂 Index:  {index_name}")
    print(f"   ❓ Query:  {query}")
    print("-" * 60)
    print("   ⏳ Processing...")
    
    credential = get_user_credential()
    
    async with (
        AzureAISearchContextProvider(
            endpoint=SEARCH_ENDPOINT,
            index_name=index_name,
            api_key=SEARCH_API_KEY,
            mode="semantic",
            top_k=5,
        ) as search,
        AzureAIAgentClient(
            project_endpoint=PROJECT_ENDPOINT,
            model_deployment_name=MODEL_DEPLOYMENT,
            credential=credential,
        ) as client,
        ChatAgent(
            chat_client=client,
            instructions=instructions,
            context_providers=[search],
        ) as agent,
    ):
        result = await agent.run(query)
        answer = result.text if hasattr(result, 'text') else str(result)
    
    # Extract structured citations from response (if framework provides them)
    citations = extract_citations(result) if show_sources else []
    
    # Format and display answer with sources
    format_answer_with_sources(answer, citations)
    
    return answer

print("✅ Semantic mode function defined")
print("📑 Source extraction enabled - inline references will be shown")

In [ ]:
# Test semantic mode with HR docs
await query_semantic_mode(
    query="What are the company values and mission?",
    index_name=SEARCH_INDEXES['hr'],
    instructions="You are an HR assistant. Answer based on company policies and documentation."
)

In [ ]:
# =============================================================================
# Test Semantic Mode with Health Benefits
# =============================================================================

await query_semantic_mode(
    query="What health insurance plans are available and what do they cover?",
    index_name=SEARCH_INDEXES['health'],
    instructions="You are a benefits specialist. Provide clear, helpful answers about health coverage."
)

## Agentic Mode Retrieval (Foundry IQ)

**Agentic mode** uses AI-powered retrieval with Foundry IQ:
- LLM plans and executes multi-step searches
- Dynamically adjusts queries based on results  
- Higher reasoning for complex questions
- Best for: Complex queries requiring synthesis across documents

**Requirements:**
- Azure OpenAI endpoint URL
- Model deployment for query planning
- Higher latency but better for complex reasoning

### ⚠️ RBAC Requirements for Agentic Mode

Agentic mode requires **Knowledge Sources** in Azure AI Search, which needs specific RBAC roles:

| Role | Purpose |
|------|---------|
| **Search Index Data Contributor** | Read/write index data |
| **Search Service Contributor** | Manage knowledge sources |

To grant these roles via Azure CLI:
```bash
# Get your user object ID
USER_ID=$(az ad signed-in-user show --query id -o tsv)

# Get your search service resource ID
SEARCH_RESOURCE_ID=$(az search service show \
    --name <your-search-service> \
    --resource-group <your-rg> \
    --query id -o tsv)

# Grant Search Index Data Contributor
az role assignment create \
    --assignee $USER_ID \
    --role "Search Index Data Contributor" \
    --scope $SEARCH_RESOURCE_ID

# Grant Search Service Contributor
az role assignment create \
    --assignee $USER_ID \
    --role "Search Service Contributor" \
    --scope $SEARCH_RESOURCE_ID
```

**If you get "Forbidden" errors**, use **Semantic Mode** instead (which works with API keys).

In [ ]:
# =============================================================================
# Agentic Mode: Query with Foundry IQ AI-Powered Retrieval
# =============================================================================

from azure.core.exceptions import HttpResponseError

async def query_agentic_mode(
    query: str,
    index_name: str,
    instructions: str = "You are a helpful assistant. Answer based on the provided context. Always mention which source document you got the information from.",
    reasoning_effort: str = "medium",
    show_sources: bool = True
):
    """
    Query using agentic mode - AI-powered retrieval with Foundry IQ.
    
    This mode:
    - LLM plans search strategy
    - Executes multi-step searches if needed
    - Adjusts queries based on intermediate results
    - Best for complex reasoning tasks
    
    Args:
        reasoning_effort: "minimal", "low", or "medium" - controls planning depth
        show_sources: If True, display source documents used for the answer
    
    Note: Requires RBAC roles on Azure AI Search:
    - Search Index Data Contributor
    - Search Service Contributor
    """
    # Header
    print("\n" + "=" * 60)
    print("🤖 AGENTIC MODE QUERY (Foundry IQ)")
    print("=" * 60)
    print(f"   📂 Index:     {index_name}")
    print(f"   🧠 Reasoning: {reasoning_effort}")
    print(f"   ❓ Query:     {query}")
    print("-" * 60)
    print("   ⏳ Processing with AI-powered retrieval...")
    
    credential = get_user_credential()
    
    try:
        async with (
            AzureAISearchContextProvider(
                endpoint=SEARCH_ENDPOINT,
                index_name=index_name,
                credential=credential,
                mode="agentic",
                azure_openai_resource_url=OPENAI_ENDPOINT,
                model_deployment_name=MODEL_DEPLOYMENT,
                retrieval_reasoning_effort=reasoning_effort,
            ) as search,
            AzureAIAgentClient(
                project_endpoint=PROJECT_ENDPOINT,
                model_deployment_name=MODEL_DEPLOYMENT,
                credential=credential,
            ) as client,
            ChatAgent(
                chat_client=client,
                instructions=instructions,
                context_providers=[search],
            ) as agent,
        ):
            result = await agent.run(query)
            answer = result.text if hasattr(result, 'text') else str(result)
        
        # Extract structured citations from response (if framework provides them)
        citations = extract_citations(result) if show_sources else []
        
        # Format and display answer with sources
        format_answer_with_sources(answer, citations)
        
        return answer
            
    except HttpResponseError as e:
        if "Forbidden" in str(e):
            print("\n" + "=" * 60)
            print("❌ RBAC PERMISSION ERROR")
            print("=" * 60)
            print("   Agentic mode requires Knowledge Sources access.")
            print("")
            print("   Required RBAC roles on Azure AI Search:")
            print("   • Search Index Data Contributor")
            print("   • Search Service Contributor")
            print("")
            print("   💡 Quick fix - use semantic mode instead:")
            print(f'   await query_semantic_mode("{query[:30]}...", "{index_name}")')
            print("=" * 60 + "\n")
        raise

print("✅ Agentic mode function defined")
print("⚠️  Note: Agentic mode requires RBAC roles on Azure AI Search")

In [ ]:
# =============================================================================
# Test Agentic Mode with Complex Query (HR Docs)
# =============================================================================

# Test with a complex query that benefits from multi-step reasoning
await query_agentic_mode(
    query="What are the key company policies around employee conduct and how do they relate to the company values?",
    index_name=SEARCH_INDEXES['hr'],
    instructions="You are an HR expert. Synthesize information from multiple policy documents to provide comprehensive answers.",
    reasoning_effort="medium"
)

## Query YAML-Configured Agents

Run queries using agent configurations from the YAML files in `./agents/`.

In [ ]:
# =============================================================================
# Query Using YAML Agent Configuration
# =============================================================================

async def query_yaml_agent(agent_name: str, query: str, mode: str = "semantic"):
    """
    Query using a YAML-configured agent.
    
    Args:
        agent_name: Name of agent from YAML configs
        query: Question to ask
        mode: "semantic" or "agentic"
    """
    config = agent_configs.get(agent_name)
    if not config:
        print(f"❌ Agent not found: {agent_name}")
        print(f"   Available: {list(agent_configs.keys())}")
        return
    
    # Get index from config
    indexes = config.get('tools', {}).get('azure_ai_search', {}).get('indexes', [])
    if not indexes:
        print(f"❌ No search indexes configured for {agent_name}")
        return
    
    index_name = indexes[0].get('name')
    instructions = config.get('instructions', '')
    
    print(f"\n📋 Agent: {agent_name}")
    print(f"   Mode: {mode}")
    
    if mode == "semantic":
        return await query_semantic_mode(query, index_name, instructions)
    else:
        return await query_agentic_mode(query, index_name, instructions)

print("✅ YAML agent query function defined")

In [38]:
# =============================================================================
# Test YAML Agents
# =============================================================================
import asyncio

# Test different agents with semantic mode
test_cases = [
    ('fulfillment-agent', 'What is the standard shipping time?'),
    ('regional-apac-agent', 'What are the APAC regional procedures?'),
    ('employee-wellness-agent', 'What health benefits are available?'),
]

print("🧪 Running agent tests...")
print("=" * 60)

for i, (agent_name, query) in enumerate(test_cases):
    if agent_name in agent_configs:
        print(f"\n[{i+1}/{len(test_cases)}] Testing: {agent_name}")
        await query_yaml_agent(agent_name, query, mode="semantic")
        
        # Small delay to allow async cleanup
        await asyncio.sleep(0.5)

print("\n" + "=" * 60)
print("✅ All agent tests completed")
print("=" * 60)


✅ ANSWER
   The Northwind Standard and Northwind Health Plus plans
   offer a variety of health benefits, although they come
   with certain limitations and exclusions.  **Northwind
   Standard Plan:** - **Covered Services:** Provides
   coverage for medical, vision, and dental services. It
   includes preventive care services and prescription drug
   coverage. - **Not Covered:** Does not cover emergency
   care, mental health and substance abuse treatment, or
   out-of-network services. You should consider alternative
   coverage options if you require these services. -
   **Additional Information:** You may need to seek pre-
   approval or authorization before receiving certain
   medical services. It’s important to use in-network
   providers to maximize benefits and minimize out-of-
   pocket expenses [source:
   /healthdocs/Northwind_Standard_Benefits_Details.pdf].
   **Northwind Health Plus Plan:** - **Covered Services:**
   Provides a wide range of health benefits and is intend

## Summary: Semantic vs Agentic Mode

| Feature | Semantic Mode | Agentic Mode (Foundry IQ) |
|---------|--------------|---------------------------|
| **Search Type** | Hybrid (keyword + vector) | AI-planned multi-step |
| **Ranking** | Semantic reranking | LLM-guided relevance |
| **Speed** | Fast | Higher latency |
| **Best For** | Direct lookups | Complex reasoning |
| **Cost** | Lower | Higher (LLM calls) |

### When to Use Each Mode

**Semantic Mode:**
- FAQ lookups
- Policy questions with direct answers
- High-volume, low-complexity queries

**Agentic Mode (Foundry IQ):**
- Multi-topic synthesis
- Comparative analysis
- Queries requiring reasoning across documents

## Direct Foundry Agent Invocation with Tracing

This section demonstrates how to create and invoke **Foundry Agents** directly using the `azure-ai-projects` SDK with:
- **Azure AI Search Tool** - Connects to our knowledge base indexes
- **OpenTelemetry Tracing** - Shows tool calls, retrieval operations, and response generation

### Why Direct Agent Invocation?
The Agent Framework's `ChatAgent` provides high-level orchestration, but for deeper visibility into:
- Tool call parameters and results
- Search queries executed against indexes
- Response generation steps

...we use the lower-level `AIProjectClient` with tracing enabled.

In [39]:
# =============================================================================
# Setup Tracing for Foundry Agent Operations
# =============================================================================
# OpenTelemetry tracing captures:
# - Agent creation and invocation
# - Tool calls (Azure AI Search queries)
# - Response generation steps
#
# Supports two modes:
# 1. Console tracing (default) - traces printed to notebook output
# 2. Azure Monitor (when APPLICATIONINSIGHTS_CONNECTION_STRING is set in .env)

# Install tracing packages if needed
# !pip install opentelemetry-sdk azure-core-tracing-opentelemetry azure-monitor-opentelemetry

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

# Check for App Insights connection string (set by notebook 05)
APP_INSIGHTS_CONN_STRING = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")

# Configure OpenTelemetry
tracer_provider = TracerProvider()

# Always add console exporter for visibility in notebook
tracer_provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))

# Optionally add Azure Monitor exporter if connection string is available
if APP_INSIGHTS_CONN_STRING:
    try:
        from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter
        azure_exporter = AzureMonitorTraceExporter(connection_string=APP_INSIGHTS_CONN_STRING)
        tracer_provider.add_span_processor(SimpleSpanProcessor(azure_exporter))
        print("✅ Azure Monitor tracing enabled")
        print(f"   Connection: ...{APP_INSIGHTS_CONN_STRING[-30:]}")
    except ImportError:
        print("⚠️  azure-monitor-opentelemetry not installed")
        print("   Run: pip install azure-monitor-opentelemetry")
else:
    print("ℹ️  Azure Monitor tracing not configured")
    print("   Set APPLICATIONINSIGHTS_CONNECTION_STRING in .env (from notebook 05)")

trace.set_tracer_provider(tracer_provider)

# Get tracer for our operations
tracer = trace.get_tracer("foundry-agent-demo")

# Enable content recording to see tool parameters and results
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"

print("\n✅ OpenTelemetry tracing configured")
print("   Traces will be printed to console showing:")
print("   • Agent operations")
print("   • Tool calls (Azure AI Search)")
print("   • Response generation")

ℹ️  Azure Monitor tracing not configured
   Set APPLICATIONINSIGHTS_CONNECTION_STRING in .env (from notebook 05)

✅ OpenTelemetry tracing configured
   Traces will be printed to console showing:
   • Agent operations
   • Tool calls (Azure AI Search)
   • Response generation


In [40]:
# =============================================================================
# Create Foundry Agent with Azure AI Search Tool
# =============================================================================
# This creates an agent in Azure AI Foundry that can query our knowledge bases
# using the Azure AI Search tool.
#
# IMPORTANT: The search connection in your Foundry project must point to the 
# same search service where your indexes exist. You can add a connection in:
# AI Foundry Portal → Management → Connected Resources → + New Connection

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AzureAISearchAgentTool,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
    PromptAgentDefinition,
    ConnectionType,
)

async def create_foundry_agent_with_search(
    agent_name: str,
    instructions: str,
    index_name: str,
    search_connection_id: str = None,
    expected_search_name: str = None,  # Optional: verify we're using the right search service
):
    """
    Create a Foundry agent with Azure AI Search tool.
    
    Args:
        agent_name: Name for the agent
        instructions: System instructions for the agent
        index_name: Azure AI Search index to connect
        search_connection_id: Project connection ID for search (optional, auto-discovered)
        expected_search_name: Expected search service name to verify connection (optional)
    
    Returns:
        Tuple of (agent, project_client, openai_client)
    """
    credential = get_user_credential()
    
    with tracer.start_as_current_span(f"create_agent_{agent_name}") as span:
        span.set_attribute("agent.name", agent_name)
        span.set_attribute("search.index", index_name)
        
        # Create the project client
        project_client = AIProjectClient(
            endpoint=PROJECT_ENDPOINT,
            credential=credential,
        )
        
        # Get connections to find search connection ID
        if not search_connection_id:
            print("📡 Finding Azure AI Search connection...")
            connections = list(project_client.connections.list())
            
            search_connections = []
            for conn in connections:
                conn_type = getattr(conn, 'type', None) or getattr(conn, 'connection_type', 'unknown')
                print(f"   Found connection: {conn.name} ({conn_type})")
                
                # Check for AI Search connection by name or type
                is_search = (
                    "search" in conn.name.lower() or 
                    conn_type == ConnectionType.AZURE_AI_SEARCH or
                    str(conn_type).lower() == "azureaisearch"
                )
                if is_search:
                    search_connections.append(conn)
            
            # If expected_search_name is provided, try to match it
            if expected_search_name and search_connections:
                for conn in search_connections:
                    if expected_search_name.lower() in conn.name.lower():
                        search_connection_id = conn.id
                        print(f"   ✅ Using (matched): {conn.name}")
                        break
            
            # Fall back to first search connection
            if not search_connection_id and search_connections:
                search_connection_id = search_connections[0].id
                print(f"   ✅ Using: {search_connections[0].name}")
                
                # Warn if there are multiple search connections
                if len(search_connections) > 1:
                    print(f"   ⚠️  Found {len(search_connections)} search connections - using first one")
                    print(f"      If your index is on a different search service, add it as a connection in AI Foundry")
        
        if not search_connection_id:
            print("⚠️  No Azure AI Search connection found in project")
            print("   Add one in AI Foundry portal → Management → Connected Resources")
            print(f"   Your search service: {SEARCH_ENDPOINT}")
            return None, None, None
        
        # Create Azure AI Search tool
        search_tool = AzureAISearchAgentTool(
            azure_ai_search=AzureAISearchToolResource(
                indexes=[
                    AISearchIndexResource(
                        project_connection_id=search_connection_id,
                        index_name=index_name,
                        query_type=AzureAISearchQueryType.SEMANTIC,
                    ),
                ]
            )
        )
        
        # Get OpenAI client for agent operations
        openai_client = project_client.get_openai_client()
        
        # Create the agent with search tool
        print(f"\n🤖 Creating agent: {agent_name}")
        print(f"   Using index: {index_name}")
        agent = project_client.agents.create_version(
            agent_name=agent_name,
            definition=PromptAgentDefinition(
                model=MODEL_DEPLOYMENT,
                instructions=instructions,
                tools=[search_tool],
            ),
        )
        print(f"   ✅ Agent created (id: {agent.id}, version: {agent.version})")
        
        return agent, project_client, openai_client

print("✅ Agent creation function defined")
print("   Uses: AIProjectClient, AzureAISearchAgentTool")
print(f"\n📋 Your search service: {SEARCH_ENDPOINT}")
print(f"   Make sure this search service is connected in AI Foundry portal")

✅ Agent creation function defined
   Uses: AIProjectClient, AzureAISearchAgentTool

📋 Your search service: https://a365-search-6uuruydd4tej6.search.windows.net
   Make sure this search service is connected in AI Foundry portal


In [41]:
# =============================================================================
# Verify Search Connection Setup
# =============================================================================
# Before creating agents, let's verify the Foundry project has access to our search service

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ConnectionType

print("🔍 SEARCH CONNECTION VERIFICATION")
print("=" * 60)
print(f"Your search service: {SEARCH_ENDPOINT}")
print(f"Expected indexes: {list(SEARCH_INDEXES.values())}")
print()

credential = get_user_credential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# List all connections
connections = list(project_client.connections.list())
search_connections = []

print("📡 Foundry project connections:")
for conn in connections:
    conn_type = getattr(conn, 'type', None) or getattr(conn, 'connection_type', 'unknown')
    marker = "🔎" if conn_type == ConnectionType.AZURE_AI_SEARCH else "  "
    print(f"   {marker} {conn.name} ({conn_type})")
    if conn_type == ConnectionType.AZURE_AI_SEARCH:
        search_connections.append(conn)

print()
if search_connections:
    print(f"✅ Found {len(search_connections)} Azure AI Search connection(s)")
    for conn in search_connections:
        print(f"   → {conn.name} (id: {conn.id[:50]}...)")
    
    # Check if our search service is connected
    our_search_name = SEARCH_ENDPOINT.replace("https://", "").replace(".search.windows.net", "")
    matched = any(our_search_name.lower() in c.name.lower() for c in search_connections)
    
    if matched:
        print(f"\n✅ Your search service '{our_search_name}' appears to be connected")
    else:
        print(f"\n⚠️  Your search service '{our_search_name}' is NOT connected!")
        print(f"   The connected search service(s): {[c.name for c in search_connections]}")
        print()
        print("   To fix this, add your search service in AI Foundry portal:")
        print("   1. Go to AI Foundry → Your Project → Management → Connected Resources")
        print("   2. Click '+ New Connection'")
        print("   3. Select 'Azure AI Search'")
        print(f"   4. Connect to: {SEARCH_ENDPOINT}")
else:
    print("❌ No Azure AI Search connections found in project")
    print("   Add one in AI Foundry portal → Management → Connected Resources")

🔍 SEARCH CONNECTION VERIFICATION
Your search service: https://a365-search-6uuruydd4tej6.search.windows.net
Expected indexes: ['hrdocs-index', 'healthdocs-index', 'agents-us', 'agents-apac']

📡 Foundry project connections:
   🔎 lab511searchl4nj2d6mf7eejc7 (ConnectionType.AZURE_AI_SEARCH)
   🔎 a365-search-connection (ConnectionType.AZURE_AI_SEARCH)
      kb-hr-benefits-assista-eejc7 (ConnectionType.REMOTE_TOOL)
      agenticdevops (ConnectionType.APPLICATION_INSIGHTS)

✅ Found 2 Azure AI Search connection(s)
   → lab511searchl4nj2d6mf7eejc7 (id: /subscriptions/63862159-43c8-47f7-9f6f-6c63d56b0e1...)
   → a365-search-connection (id: /subscriptions/63862159-43c8-47f7-9f6f-6c63d56b0e1...)

⚠️  Your search service 'a365-search-6uuruydd4tej6' is NOT connected!
   The connected search service(s): ['lab511searchl4nj2d6mf7eejc7', 'a365-search-connection']

   To fix this, add your search service in AI Foundry portal:
   1. Go to AI Foundry → Your Project → Management → Connected Resources
   2. 

In [42]:
# =============================================================================
# Invoke Foundry Agent with Tracing
# =============================================================================
# This function invokes the agent and captures all tool calls in traces

async def invoke_foundry_agent(
    agent,
    openai_client,
    project_client,
    query: str,
    show_tool_calls: bool = True,
):
    """
    Invoke a Foundry agent and show tool calls.
    
    Args:
        agent: The agent to invoke
        openai_client: OpenAI client from project
        project_client: AIProjectClient for cleanup
        query: User query
        show_tool_calls: Whether to print tool call details
    
    Returns:
        Response text
    """
    with tracer.start_as_current_span("invoke_agent") as span:
        span.set_attribute("query", query)
        span.set_attribute("agent.name", agent.name)
        
        print("\n" + "=" * 60)
        print("🔍 FOUNDRY AGENT INVOCATION")
        print("=" * 60)
        print(f"   Agent: {agent.name}")
        print(f"   Query: {query}")
        print("-" * 60)
        
        # Create a conversation
        conversation = openai_client.conversations.create(
            items=[{"type": "message", "role": "user", "content": query}],
        )
        print(f"   📝 Conversation: {conversation.id}")
        
        # Invoke agent - NOTE: Don't specify model when using agent reference
        response = openai_client.responses.create(
            conversation=conversation.id,
            extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
            input="",  # Query is in conversation
        )
        
        # Extract tool calls from response output
        if show_tool_calls:
            print("\n📊 TOOL CALLS:")
            print("-" * 40)
            tool_calls_found = False
            
            for output in response.output:
                output_type = getattr(output, 'type', None)
                
                if output_type == 'function_call':
                    tool_calls_found = True
                    print(f"   🔧 Function: {output.name}")
                    print(f"      Arguments: {output.arguments}")
                    
                elif output_type == 'azure_search_call':
                    tool_calls_found = True
                    print(f"   🔍 Azure Search Call:")
                    if hasattr(output, 'queries'):
                        for q in output.queries:
                            print(f"      Query: {q}")
                    if hasattr(output, 'results'):
                        print(f"      Results: {len(output.results)} documents")
                        
                elif output_type == 'message':
                    # Skip message outputs in tool call section
                    pass
                    
                else:
                    if output_type and output_type not in ['message']:
                        tool_calls_found = True
                        print(f"   📌 {output_type}: {output}")
            
            if not tool_calls_found:
                print("   (No explicit tool calls in response)")
        
        # Display response
        print("\n💬 RESPONSE:")
        print("-" * 40)
        print(response.output_text)
        
        # Cleanup conversation
        openai_client.conversations.delete(conversation_id=conversation.id)
        
        return response.output_text

print("✅ Agent invocation function defined")
print("   Captures: conversations, tool calls, responses")

✅ Agent invocation function defined
   Captures: conversations, tool calls, responses


In [43]:
# =============================================================================
# Test: Create and Invoke HR Agent with Search
# =============================================================================
# Creates an agent connected to the hrdocs-index and queries it
# Note: We specifically use 'a365-search-connection' which should point to our search service

# Create the HR agent - use expected_search_name to match our search service connection
hr_agent, project_client, openai_client = await create_foundry_agent_with_search(
    agent_name="hr-knowledge-agent",
    instructions="""You are an HR specialist for Zava company. 
Answer questions about company policies, benefits, and procedures.
Always cite the source documents when providing information.
Be concise but thorough.""",
    index_name=SEARCH_INDEXES['hr'],  # hrdocs-index
    expected_search_name="a365-search",  # Match our search service connection name
)

if hr_agent:
    # Test query
    await invoke_foundry_agent(
        hr_agent,
        openai_client,
        project_client,
        query="What are the company's policies on remote work and flexible hours?",
        show_tool_calls=True,
    )
    
    # Cleanup
    # project_client.agents.delete_version(agent_name=hr_agent.name, agent_version=hr_agent.version)
    # print("\n🗑️  Agent cleaned up")

📡 Finding Azure AI Search connection...
   Found connection: lab511searchl4nj2d6mf7eejc7 (ConnectionType.AZURE_AI_SEARCH)
   Found connection: a365-search-connection (ConnectionType.AZURE_AI_SEARCH)
   Found connection: kb-hr-benefits-assista-eejc7 (ConnectionType.REMOTE_TOOL)
   Found connection: agenticdevops (ConnectionType.APPLICATION_INSIGHTS)
   ✅ Using (matched): a365-search-connection

🤖 Creating agent: hr-knowledge-agent
   Using index: hrdocs-index
   ✅ Agent created (id: hr-knowledge-agent:2, version: 2)
{
    "name": "create_agent_hr-knowledge-agent",
    "context": {
        "trace_id": "0xdfe595c42fc4b00252a1548a09892baf",
        "span_id": "0xaf9b8cafbf746704",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-01-21T00:44:40.795397Z",
    "end_time": "2026-01-21T00:44:41.261267Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "agent.name": "hr-knowledge-agent",
        "

In [44]:
# =============================================================================
# Test: Create and Invoke Health Benefits Agent
# =============================================================================
# Creates an agent connected to the healthdocs-index and queries it

health_agent, project_client, openai_client = await create_foundry_agent_with_search(
    agent_name="health-benefits-agent",
    instructions="""You are a health benefits specialist.
Answer questions about health insurance plans, coverage, and benefits.
Provide specific details from the policy documents.
If information is not available, say so clearly.""",
    index_name=SEARCH_INDEXES['health'],  # healthdocs-index
    expected_search_name="a365-search",  # Match our search service connection name
)

if health_agent:
    # Test query
    await invoke_foundry_agent(
        health_agent,
        openai_client,
        project_client,
        query="What mental health services are covered under the health insurance plan?",
        show_tool_calls=True,
    )
    
    # Cleanup
    # project_client.agents.delete_version(agent_name=health_agent.name, agent_version=health_agent.version)
    # print("\n🗑️  Agent cleaned up")

📡 Finding Azure AI Search connection...
   Found connection: lab511searchl4nj2d6mf7eejc7 (ConnectionType.AZURE_AI_SEARCH)
   Found connection: a365-search-connection (ConnectionType.AZURE_AI_SEARCH)
   Found connection: kb-hr-benefits-assista-eejc7 (ConnectionType.REMOTE_TOOL)
   Found connection: agenticdevops (ConnectionType.APPLICATION_INSIGHTS)
   ✅ Using (matched): a365-search-connection

🤖 Creating agent: health-benefits-agent
   Using index: healthdocs-index
   ✅ Agent created (id: health-benefits-agent:1, version: 1)
{
    "name": "create_agent_health-benefits-agent",
    "context": {
        "trace_id": "0x8ff96fa64edf41e6fd565ce15b528702",
        "span_id": "0x1f1a012a2e38e36d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-01-21T00:44:45.739770Z",
    "end_time": "2026-01-21T00:44:46.300621Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "agent.name": "health-benefits-a

### Understanding the Traces

When you run the agent invocations above, OpenTelemetry captures:

| Span | Description |
|------|-------------|
| `create_agent_*` | Agent creation with search tool configuration |
| `invoke_agent` | Full agent invocation including tool calls |
| Azure AI Search calls | Queries sent to the search index |
| Response generation | LLM processing and response creation |

**Tool Calls in Response:**
- `azure_search_call` - Search queries against the index
- `function_call` - Custom function invocations
- `message` - Final response text

For production monitoring, connect to **Azure Monitor Application Insights**:
```python
from azure.monitor.opentelemetry import configure_azure_monitor
app_insights_conn = project_client.telemetry.get_application_insights_connection_string()
configure_azure_monitor(connection_string=app_insights_conn)
```